# Pilot P3 + P4: FQ-QAT Stability and QUAIL Baseline

## IMPORTANT: Run on SINGLE T4 GPU (not dual)

- **P3**: Tests whether FQ-QAT + NPO training is stable on forget01
- **P4**: Establishes the Q-INT4 score for the QUAIL baseline
- **Estimated total time**: 60-90 minutes
- **Results saved to**: /kaggle/working/results/

### Before running:
1. Set accelerator to **GPU T4 x1** (single GPU)
2. Ensure **Internet** is toggled ON
3. Replace `YOUR_HF_TOKEN` with your actual HuggingFace token

## Cell 1: Setup and Dependencies

In [ ]:
import os
import subprocess

# ── Configuration ────────────────────────────────────────────
HF_TOKEN = "YOUR_HF_TOKEN"  # <-- REPLACE THIS
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["TRANSFORMERS_NO_FLASH_ATTN"] = "1"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = "/kaggle/working/fq-ssiuu/src:" + os.environ.get("PYTHONPATH", "")

print("=== Cloning Repositories ===")

# Clone fq-ssiuu
if not os.path.exists("/kaggle/working/fq-ssiuu"):
    subprocess.run(
        "git clone --branch pilot/p3-p4 https://github.com/ibn-e-Muhammad/fq-ssiuu.git /kaggle/working/fq-ssiuu",
        shell=True, check=True,
    )
else:
    subprocess.run(
        "cd /kaggle/working/fq-ssiuu && git fetch origin && git checkout pilot/p3-p4 && git pull origin pilot/p3-p4",
        shell=True, check=True,
    )

# Clone OpenUnlearning
if not os.path.exists("/kaggle/working/open-unlearning"):
    subprocess.run(
        "git clone --branch fq-ssiuu-integration https://github.com/ibn-e-Muhammad/open-unlearning.git /kaggle/working/open-unlearning",
        shell=True, check=True,
    )
else:
    subprocess.run(
        "cd /kaggle/working/open-unlearning && git fetch origin && git checkout fq-ssiuu-integration",
        shell=True, check=True,
    )

print("\n=== Patching Dependencies ===")

# Remove toxic version pins
subprocess.run("sed -i '/torch==/d' /kaggle/working/open-unlearning/requirements.txt", shell=True)
subprocess.run("sed -i '/torchvision==/d' /kaggle/working/open-unlearning/requirements.txt", shell=True)
subprocess.run("sed -i '/bitsandbytes==/d' /kaggle/working/open-unlearning/requirements.txt", shell=True)
subprocess.run("sed -i 's/hf_vlms/huggingface/g' /kaggle/working/open-unlearning/src/evals/lm_eval.py", shell=True)

print("\n=== Purging Corrupted System Numpy ===")
# Kaggle's pip leaves behind old C-extensions when upgrading/downgrading numpy, 
# causing 'cannot import name _center' or 'numpy.char' ModuleNotFoundErrors.
# We MUST physically delete the corrupted packages before installing clean ones.
subprocess.run("pip uninstall numpy scipy scikit-learn -y -q", shell=True)
subprocess.run("rm -rf /usr/local/lib/python3.12/dist-packages/numpy*", shell=True)
subprocess.run("rm -rf /usr/local/lib/python3.12/dist-packages/scipy*", shell=True)
subprocess.run("rm -rf /usr/local/lib/python3.12/dist-packages/scikit_learn*", shell=True)
subprocess.run("rm -rf /usr/local/lib/python3.12/dist-packages/sklearn*", shell=True)
subprocess.run("rm -rf /opt/conda/lib/python3.12/site-packages/numpy*", shell=True)
subprocess.run("rm -rf /opt/conda/lib/python3.12/site-packages/scipy*", shell=True)

print("\n=== Installing Clean Dependencies ===")
# Install a clean, stable pre-2.0 matrix
subprocess.run("pip install numpy==1.26.4 scipy==1.13.1 scikit-learn==1.4.2 -q", shell=True)

# Install core packages
subprocess.run("pip install -e /kaggle/working/open-unlearning -q", shell=True)
subprocess.run("pip install peft lm-eval==0.4.4 bitsandbytes==0.45.5 captum -q", shell=True)
subprocess.run("pip uninstall timm flash-attn -y -q 2>/dev/null || true", shell=True)

# Install fq-ssiuu (no deps)
subprocess.run("pip install -e /kaggle/working/fq-ssiuu --no-deps -q", shell=True)

print("\n=== Downloading Retain Logs ===")
subprocess.run(
    "cd /kaggle/working/open-unlearning && python setup_data.py --eval_logs",
    shell=True,
)

retain_logs = "/kaggle/working/open-unlearning/saves/eval/tofu_Llama-3.2-1B-Instruct_retain90/TOFU_EVAL.json"
if not os.path.exists(retain_logs):
    print("setup_data.py didn't produce the file. Trying direct download...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="open-unlearning/eval",
        allow_patterns="*.json",
        repo_type="dataset",
        local_dir="/kaggle/working/open-unlearning/saves/eval",
    )

assert os.path.exists(retain_logs), f"FATAL: Retain logs not found at {retain_logs}"
print(f"✅ Retain logs ready: {retain_logs}")
print("\n=== Setup Complete ===")

## Cell 2: Verify Setup

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

import bitsandbytes
print(f"bitsandbytes: {bitsandbytes.__version__}")

import peft
print(f"peft: {peft.__version__}")

import captum
print(f"captum: {captum.__version__}")

print("\n✅ All checks passed. Ready for P3 and P4.")

## Cell 3: Run Pilot P3 (FQ-QAT + NPO Stability Test)

In [ ]:
import os, sys, subprocess

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["TRANSFORMERS_NO_FLASH_ATTN"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

OU_PATH = "/kaggle/working/open-unlearning"
OUTPUT_DIR = "/kaggle/working/results/pilot_p3"

print("=== PILOT P3: FQ-QAT + NPO STABILITY TEST ===\n")

result = subprocess.run(
    [
        sys.executable,
        "/kaggle/working/fq-ssiuu/experiments/pilot_p3_fqat_stability.py",
        "--open_unlearning_path", OU_PATH,
        "--drive_output", OUTPUT_DIR,
    ],
    capture_output=False,
    text=True,
    env=os.environ,
)

print(f"\n=== P3 Exit Code: {result.returncode} ===")

## Cell 4: Run Pilot P4 (QUAIL Baseline)

In [ ]:
import os, sys, subprocess

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["TRANSFORMERS_NO_FLASH_ATTN"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

OU_PATH = "/kaggle/working/open-unlearning"
OUTPUT_DIR = "/kaggle/working/results/pilot_p4"

print("=== PILOT P4: QUAIL BASELINE ===\n")

result = subprocess.run(
    [
        sys.executable,
        "/kaggle/working/fq-ssiuu/experiments/pilot_p4_quail_baseline.py",
        "--open_unlearning_path", OU_PATH,
        "--drive_output", OUTPUT_DIR,
    ],
    capture_output=False,
    text=True,
    env=os.environ,
)

print(f"\n=== P4 Exit Code: {result.returncode} ===")